In [ ]:
# name: sonali vishal pawar
# title: AI Resume Screening System with LangChain and LangSmith

In [ ]:
# Project Summary:
 # This project is an AI-powered Resume Screening System developed using Python, LangChain, and LangSmith. The system analyzes resumes based on a given job description, extracts candidate skills and experience, compares them with required job skills, and generates a match score with explanation. The project also demonstrates LangSmith tracing for monitoring and debugging the AI pipeline.

In [ ]:
# Objective:
# The main objective of this project is to automate the resume screening process using Generative AI techniques. The system helps recruiters identify suitable candidates efficiently by evaluating resumes based on skills, experience, and job requirements.

In [ ]:
# Features:
• Resume Skill Extraction
• Experience Identification
• Resume-to-Job Matching
• Candidate Scoring System
• Explainable AI Output
• LangChain Pipeline Implementation
• LangSmith Tracing and Monitoring

In [ ]:
#Technologies Used:
• Python
• LangChain
• LangSmith
• Hugging Face
• Google Colab

In [ ]:
# Workflow:
Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing

In [1]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=60
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
# Create Project Folder

In [3]:
import os

folders = [
    "AI-Resume-Screening-System/data",
    "AI-Resume-Screening-System/screenshots"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders Created Successfully!")

Folders Created Successfully!


In [4]:
# Create Resume Data

In [5]:
strong_resume = """
John Doe

5 years of experience in Data Science and Machine Learning.

Skills:
Python, SQL, Machine Learning, Deep Learning, NLP, Docker, AWS

Tools:
TensorFlow, Scikit-learn, Pandas
"""

average_resume = """
Jane Smith

2 years experience in Python and Data Analysis.

Skills:
Python, SQL, Excel, Machine Learning

Tools:
Pandas, NumPy
"""

weak_resume = """
Alex Brown

Fresher candidate.

Skills:
MS Excel, Communication

No AI experience.
"""

job_description = """
We are hiring a Data Scientist.

Required Skills:
Python
Machine Learning
Deep Learning
SQL
NLP
Docker
AWS

Experience:
3+ years preferred.
"""

In [6]:
# Save Files

In [7]:
with open("/content/AI-Resume-Screening-System/data/strong_resume.txt", "w") as f:
    f.write(strong_resume)

with open("/content/AI-Resume-Screening-System/data/average_resume.txt", "w") as f:
    f.write(average_resume)

with open("/content/AI-Resume-Screening-System/data/weak_resume.txt", "w") as f:
    f.write(weak_resume)

with open("/content/AI-Resume-Screening-System/data/job_description.txt", "w") as f:
    f.write(job_description)

print("Files Saved Successfully!")

Files Saved Successfully!


In [8]:
# Add LangSmith Key

In [9]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_API_KEY"] = "-------------------------"

os.environ["LANGCHAIN_PROJECT"] = "AI-Resume-Screening-System"

In [10]:
# Import LangChain

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

In [12]:
# Create Prompt Template

In [13]:
prompt = PromptTemplate(
    input_variables=["resume"],

    template="""
Extract:
1. Skills
2. Experience

Resume:
{resume}
"""
)

In [14]:
# Create Resume Extraction Function

In [15]:
def extract_resume_details(prompt_value):

    text = prompt_value.to_string()

    resume = text.lower()

    skills = []

    skill_keywords = [
        "Python",
        "SQL",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Docker",
        "AWS",
        "TensorFlow",
        "Pandas",
        "Scikit-learn",
        "Excel"
    ]

    for skill in skill_keywords:

        if skill.lower() in resume:
            skills.append(skill)

    experience = "Not Found"

    if "5 years" in resume:
        experience = "5 years"

    elif "2 years" in resume:
        experience = "2 years"

    elif "fresher" in resume:
        experience = "Fresher"

    return {
        "Skills": skills,
        "Experience": experience
    }

In [16]:
# Create LangChain Pipeline

In [17]:
chain = prompt | RunnableLambda(extract_resume_details)

In [18]:
# Load Strong Resume

In [19]:
resume = open(
    "/content/AI-Resume-Screening-System/data/strong_resume.txt"
).read()

print(resume)


John Doe

5 years of experience in Data Science and Machine Learning.

Skills:
Python, SQL, Machine Learning, Deep Learning, NLP, Docker, AWS

Tools:
TensorFlow, Scikit-learn, Pandas



In [20]:
# Run Extraction

In [21]:
result = chain.invoke({
    "resume": resume
})

print(result)

{'Skills': ['Python', 'SQL', 'Machine Learning', 'Deep Learning', 'NLP', 'Docker', 'AWS', 'TensorFlow', 'Pandas', 'Scikit-learn'], 'Experience': '5 years'}


In [22]:
# Required Skills

In [23]:
required_skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "Docker",
    "AWS"
]

In [24]:
# Scoring Function

In [25]:
def calculate_score(candidate_skills, required_skills):

    matched = []

    for skill in candidate_skills:

        if skill in required_skills:
            matched.append(skill)

    score = int(
        (len(matched) / len(required_skills)) * 100
    )

    return score, matched

In [26]:
# Calculate Score

In [27]:
score, matched_skills = calculate_score(
    result["Skills"],
    required_skills
)

print("Score:", score)

print("Matched Skills:", matched_skills)

Score: 100
Matched Skills: ['Python', 'SQL', 'Machine Learning', 'Deep Learning', 'NLP', 'Docker', 'AWS']


In [28]:
# Missing Skills + Explanation

In [29]:
missing_skills = list(
    set(required_skills) - set(matched_skills)
)

print("Missing Skills:", missing_skills)

print()

if score >= 80:
    print("Explanation: Strong candidate for the role.")

elif score >= 50:
    print("Explanation: Average candidate with partial matching skills.")

else:
    print("Explanation: Weak candidate with limited required skills.")

Missing Skills: []

Explanation: Strong candidate for the role.


In [30]:
# Run for Average Resume

In [32]:
resume = open(
    "/content/AI-Resume-Screening-System/data/average_resume.txt"
).read()

result = chain.invoke({
    "resume": resume
})

print(result)

{'Skills': ['Python', 'SQL', 'Machine Learning', 'Pandas', 'Excel'], 'Experience': '2 years'}


In [33]:
resume = open(
    "/content/AI-Resume-Screening-System/data/weak_resume.txt"
).read()

result = chain.invoke({
    "resume": resume
})

print(result)

{'Skills': ['Excel'], 'Experience': 'Fresher'}


In [34]:
score, matched_skills = calculate_score(
    result["Skills"],
    required_skills
)

print("Score:", score)

print("Matched Skills:", matched_skills)

Score: 0
Matched Skills: []


In [35]:
missing_skills = list(
    set(required_skills) - set(matched_skills)
)

print("Missing Skills:", missing_skills)

if score >= 80:
    print("Explanation: Strong candidate")

elif score >= 50:
    print("Explanation: Average candidate")

else:
    print("Explanation: Weak candidate")

Missing Skills: ['Deep Learning', 'SQL', 'Machine Learning', 'NLP', 'AWS', 'Docker', 'Python']
Explanation: Weak candidate


In [ ]:
# Conclusion:
# This project demonstrates how Generative AI can simplify and improve the recruitment process. By integrating LangChain and LangSmith, the system provides an efficient, explainable, and traceable resume evaluation pipeline that can assist recruiters in selecting suitable candidates quickly and accurately.